In [ ]:
from pathlib import Path
from loguru import logger
import pandas as pd
from datetime import datetime

Read in the file

In [ ]:
import tomllib

configfile = Path("../config.toml").resolve()
with configfile.open("rb") as f:
    config = tomllib.load(f)
processed = Path("../data/processed")
datafile = processed / config["inputpath"]
if not datafile.exists():
    logger.warning(
        f"{datafile} does not exist. Maybe first run src/preprocess.py, or check the timestamp!"
    )

In [ ]:
df = pd.read_csv(datafile, parse_dates=["timestamp"])
df.head()

Sometimes, author names have a tilde in front of them, allong with some unicode. Let's clean that.

In [ ]:
import re

clean_tilde = r"^~\u202f"
df["author"] = df["author"].apply(lambda x: re.sub(clean_tilde, "", x))

Let's check how many unique authors we have

In [ ]:
len(df.author.unique())
df.author.unique()

Let's add age and gender

In [ ]:
ages = {
    'Loïs Vriens': 24,
    'Kris van den Oever': 30,
    'Cheryl Van Den Oever': 28,
    'Regina Veld': 54,
    'Gijs van den Oever': 34,
    'Melissa': 32,
    'Ties Van Den Oever': 33,
    'Bram Van Den Oever' : 37,
    'Allard' : 36,
    'Jos Vriens' : 60,
    'Anouk Van Den Oever-blankers' : 27,
    'Wendy Van Den Oever' : 35
}

genders = {
    'Loïs Vriens': 'f',
    'Kris van den Oever': 'm',
    'Cheryl Van Den Oever': 'f',
    'Regina Veld': 'f',
    'Gijs van den Oever': 'm',
    'Melissa': 'f',
    'Ties Van Den Oever': 'm',
    'Bram Van Den Oever' : 'm',
    'Allard' : 'm',
    'Jos Vriens' : 'm',
    'Anouk Van Den Oever-blankers' : 'm',
    'Wendy Van Den Oever' : 'f'
}

# Voeg leeftijd toe
df['age'] = df['author'].map(ages)

# Voeg geslacht toe
df['gender'] = df['author'].map(genders)

df.head()

In [ ]:
import json
from wa_analyzer.humanhasher import humanize

authors = df.author.unique()
anon = {k: humanize(k) for k in authors}
# we save a reference file so we can look up the original author names if we want to
reference_file = processed / "anon_reference.json"

with open(reference_file, "w") as f:
    # invert the dictionary:
    ref = {v: k for k, v in anon.items()}
    # sort alphabetically:
    ref_sorted = {k: ref[k] for k in sorted(ref.keys())}
    # save as json:
    json.dump(ref_sorted, f)

assert len(anon) == len(authors), "you lost some authors!"

In [ ]:
# set anonymous authors
df["anon_author"] = df.author.map(anon)
df

In [ ]:
# drop author
df.drop(columns=["author"], inplace=True)
df

In [ ]:
# rename author colomn
df.rename(columns={"anon_author": "author"}, inplace=True)
df.head()

In my case, the first line is a header, saying messages are encrypted. Let's remove that. Your data might be different, so double check if you also want to remove the first line!

In [ ]:
df = df.drop(index=[0])

let's check:

In [ ]:
df.head()

Let's create a timestamp for a new, unique, filename.

In [ ]:
now = datetime.now().strftime("%Y%m%d-%H%M%S")
output = processed / f"whatsapp-{now}.csv"
output

Let's save the file both as a csv and as a parquet file.
Parquet has some advantages:
- its about 100x faster to read and write
- datatypes are preserved (eg the timestamp type). You will loose this in a csv file.
- file size is much smaller

The advantage of csv is that you can easily peak at the data in a text editor.

In [ ]:
df.to_csv(output, index=False)
df.to_parquet(output.with_suffix(".parq"), index=False)

Now, go to `config.toml` and change the name by "current" to the parquet file you just created.
This makes it easier to use the same file everywhere, without the need to continuously retype the name if you change it.